# NHANES BMI ↔ LDL-C correlation — overview

Estimates the correlation coefficient between body-mass index (BMI)
and low-density-lipoprotein cholesterol (LDL-C) in U.S. adults, for
use in `vivarium_csu_mace_rct`'s Model 2 `RiskCorrelation` component.

Two analyses live here. The first reports the correlation as
observed in the post-treatment NHANES population — what the
simulation will see if it initializes simulants from the GBD
exposure distribution (which already bakes in current statin
therapy). The second deletes the treatment effect using a
typical-efficacy assumption to recover the underlying biological
correlation, and compares.

## Headline results

**As observed (post-treatment).** Source: NHANES 1999–2018, weighted
Spearman, MEC weights pooled across cycles.

| Population                              | ρ (weighted Spearman) | n      |
| :---                                    |                  ---: | ---:   |
| All adults (age ≥ 20)                   |                +0.09  | 22,414 |
| **Trial-eligible (age 65–80)**          |           **−0.09**   |  4,005 |
| vnuc reference (age-invariant)          |                +0.11  |    n/a |

**Treatment-deleted (counterfactual untreated).** Same NHANES data;
observed LDL-C divided by `(1 − δ)` for participants reporting they
currently take prescription cholesterol medication, with δ = 0.30
(representative real-world statin LDL-C-lowering fraction).

| Population                              | ρ (observed) | ρ (untreated) |   Δ   |
| :---                                    |         ---: |          ---: |  ---: |
| All adults (age ≥ 20)                   |       +0.09  |       +0.13   | +0.04 |
| Trial-eligible (age 65–80)              |       −0.09  |       −0.06   | +0.04 |

## What this means

- **Treatment is a major driver of the negative correlation in older
  adults, but not the only one.** Even after deleting a generous
  30 % LDL-C reduction, the trial-eligible (65–80) Spearman is still
  negative (−0.06). Sweeping δ from 0 to 0.55 confirms the trial-band
  value crosses zero only at unrealistically high efficacies (~0.55).
- **Vnuc's all-ages reference (+0.11) is very close to the
  treatment-deleted pooled value (+0.13).** This strongly suggests
  vnuc's coefficient reflects a treatment-naive or older NHANES
  sample, not the current as-observed population.
- **The BMI ↔ LDL-C correlation remains strongly age-dependent under
  either analysis** — positive in young adults, near zero in middle
  age, negative in older adults. Treatment deletion shifts the curve
  upward by a roughly constant ~0.04–0.07 across age bands but does
  not eliminate the age trend.

## Recommendation

Use **ρ = −0.09** for the BMI ↔ LDL-C entry in mace_rct Model 2's
`RiskCorrelation` matrix. Rationale:

1. mace_rct initializes BMI and LDL-C exposures from the GBD risk
   distributions, which are post-treatment. Matching the simulation's
   initialization to the as-observed correlation keeps the joint
   distribution self-consistent.
2. Resmetirom, the trial intervention, is layered on top of whatever
   baseline lipid-lowering therapy exists in the GBD distribution.
   There's no clean way for the simulation to "know" which simulants
   are already on statins, so trying to use the untreated correlation
   would create a propensity assignment that doesn't match the
   initialized exposures.
3. The treatment-deleted value (−0.06) is within ~0.04 of the
   observed value, so the choice is unlikely to materially change
   Model 2 results either way.

Two issues this surfaces for follow-up:

1. The vivarium_research SBP/LDL-C/FPG/BMI correlation spec calls
   the BMI ↔ LDL-C pair "approximately age-invariant" — NHANES
   (under either treatment-status assumption) contradicts that.
   Worth re-checking the other "simplified" pairs.
2. A long-running simulation would need an age-stratified
   correlation table; the single-coefficient approximation is OK
   only because Model 2's 5-year window confines simulants to ~1–2
   age bins.

## Notebooks

- `01_correlation.ipynb` — primary analysis: per-cycle stability,
  age-band stratification, sex stratification, trial-eligible
  subset, recommendation. As-observed values only.
- `02_treatment_deletion.ipynb` — sensitivity: download NHANES BPQ
  files, identify cholesterol-medication users, apply a δ-fractional
  LDL-C deletion to recover counterfactual untreated values, and
  re-compute every correlation. Sweeps δ from 0 to 0.55 to confirm
  the qualitative pattern doesn't depend on the assumed efficacy.

Both notebooks are re-executable in place; outputs are committed.

## Data

- **Risk values** (`BMXBMI`, `LBDLDL`, demographics, MEC weights):
  NHANES continuous cycles 1999–2018, prepared parquets at
  `../data/derived/{cycle}.parquet` by the `nhanes_vivarium_risk_demo`
  project. 2017–2020 pre-pandemic and 2021–2022 cycles excluded —
  LDL-C is missing from the prepared parquets for those.
- **Treatment status** (`BPQ100D`, "now taking prescription cholesterol
  medication"): NHANES BPQ files, downloaded on first run of
  `02_treatment_deletion.ipynb` to `../data/raw/nhanes/{cycle}/`.